<a href="https://colab.research.google.com/github/10kshaodow/avian_binary_flight_classifier/blob/main/avian_flight_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avian Flight Image Classifier
**Binary classification:** `in_flight` vs `not_in_flight`

### Pipeline
1. Mount Drive & install deps
2. Dataset setup & augmentation
3. Train ResNet-50, EfficientNet-B3, ViT-B/16 (DINO)
4. Evaluate — accuracy, AUC, F1, confusion matrix
5. Grad-CAM / Attention rollout interpretability

### Expected folder structure in your Google Drive:
```
MyDrive/avian_dataset/
├── train/
│   ├── in_flight/       ← your in-flight images
│   └── not_in_flight/   ← your not-in-flight images
├── val/
│   ├── in_flight/
│   └── not_in_flight/
└── test/
    ├── in_flight/
    └── not_in_flight/
```
> If your data isn't split yet, set `AUTO_SPLIT = True` in Section 2 and point `RAW_DATA_DIR` at a flat two-class folder.

**YOLO** is used here as a preprocessing step to localize the bird within each image before classification. Instead of training the classifier on full scenes, which can encourage it to rely on background cues like sky, branches, or water, YOLO detects the bird and provides a bounding box around it. The image is then cropped around that detected region, allowing the downstream classifier to focus more directly on the bird’s shape and pose rather than the surrounding context. This makes the classification pipeline more aligned with the actual task of recognizing whether the bird itself is in flight or not.

## 1 · Setup

In [ ]:
# Install extra deps (timm for ViT, grad-cam)
!pip install -q timm grad-cam scikit-learn matplotlib seaborn
!pip -q install ultralytics pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, random, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
import timm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from PIL import Image

from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 · Configuration — edit this cell

In [ ]:
# Path to your dataset root in Drive
DATA_DIR = '/content/drive/Shareddrives/Project_CMPE679_final/Project/avian_flight_stage_dataset'

# Set True if data is NOT already split into train/val/test subfolders
AUTO_SPLIT = True
RAW_DATA_DIR = '/content/drive/Shareddrives/Project_CMPE679_final/Project/avian_flight_stage_dataset'
SPLIT_RATIOS = (0.70, 0.15, 0.15)   # train / val / test

# Which models to train (set False to skip)
RUN_RESNET       = False
RUN_EFFICIENTNET = False
RUN_VIT          = True

# Training hyperparameters
IMG_SIZE       = 224
BATCH_SIZE     = 32
EPOCHS         = 20
LR             = 1e-4
WEIGHT_DECAY   = 1e-2
LABEL_SMOOTHING = 0.1
PATIENCE       = 7

# Output directory (saved to Drive)
OUT_DIR = '/content/drive/Shareddrives/Project_CMPE679_final/Project/avian_outputs'

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import shutil
import os

split_dir = '/content/drive/Shareddrives/Project_CMPE679_final/Project/avian_outputs/split_dataset'

if os.path.exists(split_dir):
    shutil.rmtree(split_dir)
    print("Deleted old split dataset")
else:
    print("No old split dataset found")

## 3 · YOLO Preprocesssing

In [ ]:
from tqdm.auto import tqdm
SRC_ROOT = RAW_DATA_DIR

# Output = cropped dataset

DST_ROOT = "/content/drive/Shareddrives/Project_CMPE679_final/Project/avian_binary_dataset_cropped"

CLASS_NAMES = ["in_flight", "not_in_flight"]

# YOLO / crop settings

CONF_THRESH = 0.20

PADDING_FRAC = 0.15

FALLBACK_TO_ORIGINAL = True

OVERWRITE_DST = True

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

BIRD_CLASS_ID = 14   # COCO bird class

DEVICE_YOLO = 0 if torch.cuda.is_available() else "cpu"

yolo_model = YOLO("yolov8n.pt")

print("YOLO device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def is_image_file(filename):
    return filename.lower().endswith(IMG_EXTS)

def make_output_dirs(dst_root, class_names, overwrite=True):
    if overwrite and os.path.exists(dst_root):
        shutil.rmtree(dst_root)
    for cls in class_names:
        os.makedirs(os.path.join(dst_root, cls), exist_ok=True)

def add_padding(x1, y1, x2, y2, width, height, pad_frac=0.15):
    bw = x2 - x1
    bh = y2 - y1
    pad_w = bw * pad_frac
    pad_h = bh * pad_frac

    x1 = max(0, int(x1 - pad_w))
    y1 = max(0, int(y1 - pad_h))
    x2 = min(width, int(x2 + pad_w))
    y2 = min(height, int(y2 + pad_h))
    return x1, y1, x2, y2

def best_bird_box(result, conf_thresh=0.20):
    if result.boxes is None or len(result.boxes) == 0:
        return None

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)

    best = None
    best_conf = -1.0

    for box, conf, cls_id in zip(boxes, confs, classes):
        if cls_id == BIRD_CLASS_ID and conf >= conf_thresh and conf > best_conf:
            best_conf = conf
            best = (*box.tolist(), float(conf))

    return best

make_output_dirs(DST_ROOT, CLASS_NAMES, overwrite=OVERWRITE_DST)

stats = {"cropped": 0, "fallback_original": 0, "skipped": 0, "errors": 0}

for cls in CLASS_NAMES:
    src_dir = os.path.join(SRC_ROOT, cls)
    dst_dir = os.path.join(DST_ROOT, cls)

    files = [f for f in os.listdir(src_dir) if is_image_file(f)]
    print(f"\nProcessing {cls}: {len(files)} images")

    for fname in tqdm(files):
        src_path = os.path.join(src_dir, fname)
        dst_path = os.path.join(dst_dir, fname)

        try:
            image = Image.open(src_path).convert("RGB")
            width, height = image.size

            result = yolo_model.predict(
                source=src_path,
                conf=CONF_THRESH,
                verbose=False,
                device=DEVICE_YOLO
            )[0]

            box = best_bird_box(result, conf_thresh=CONF_THRESH)

            if box is not None:
                x1, y1, x2, y2, _ = box
                x1, y1, x2, y2 = add_padding(
                    x1, y1, x2, y2, width, height, pad_frac=PADDING_FRAC
                )
                crop = image.crop((x1, y1, x2, y2))
                crop.save(dst_path)
                stats["cropped"] += 1
            else:
                if FALLBACK_TO_ORIGINAL:
                    shutil.copy2(src_path, dst_path)
                    stats["fallback_original"] += 1
                else:
                    stats["skipped"] += 1

        except Exception as e:
            stats["errors"] += 1
            print(f"Error on {src_path}: {e}")

print("\nYOLO preprocessing complete.")
print(stats)


In [ ]:
RAW_DATA_DIR = DST_ROOT

## 4 · Data Loaders & Augmentation

In [ ]:
# Auto-split if needed
import shutil
import random

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def auto_split(raw_dir, out_dir, ratios):
    """
    Split a flat class-folder dataset into train/val/test.

    Expected input:
    raw_dir/
        in_flight/
        not_in_flight/
        transition/
    """
    classes = sorted([
        d for d in os.listdir(raw_dir)
        if os.path.isdir(os.path.join(raw_dir, d))
    ])

    print(f'Classes found: {classes}')

    # clear old split if it exists
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    for cls in classes:
        cls_path = os.path.join(raw_dir, cls)

        imgs = [
            f for f in os.listdir(cls_path)
            if os.path.isfile(os.path.join(cls_path, f))
            and f.lower().endswith(IMG_EXTS)
        ]

        random.shuffle(imgs)
        n = len(imgs)

        n_train = int(n * ratios[0])
        n_val = int(n * ratios[1])

        splits = {
            'train': imgs[:n_train],
            'val': imgs[n_train:n_train + n_val],
            'test': imgs[n_train + n_val:]
        }

        for split, files in splits.items():
            dest = os.path.join(out_dir, split, cls)
            os.makedirs(dest, exist_ok=True)

            for f in files:
                shutil.copy2(os.path.join(cls_path, f), os.path.join(dest, f))

            print(f'{split}/{cls}: {len(files)} images')

if AUTO_SPLIT:
    DATA_DIR = os.path.join(OUT_DIR, 'split_dataset')
    auto_split(RAW_DATA_DIR, DATA_DIR, SPLIT_RATIOS)
    print('Split complete!')

In [ ]:
# Transforms
# Training: aggressive augmentation to fight background shortcuts
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.01),
    transforms.RandomGrayscale(p=0.125),# was 0.05
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.28, scale=(0.02, 0.06), ratio=(0.3, 3.3)),
    ])


eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Datasets
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'), eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), eval_tf)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f'Classes: {CLASS_NAMES}')
print(f'Num classes: {NUM_CLASSES}')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Weighted loss to handle class imbalance
targets = torch.tensor(train_ds.targets)
class_counts = torch.bincount(targets, minlength=NUM_CLASSES).float()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print("Class counts:", class_counts.tolist())
print("Class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(DEVICE),
    label_smoothing=LABEL_SMOOTHING
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(train_ds.classes)
print(torch.bincount(torch.tensor(train_ds.targets)))

# Sanity check: show a batch
def show_batch(loader, n=8):
    imgs, labels = next(iter(loader))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    imgs = (imgs * std + mean).clamp(0, 1)

    n = min(n, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(2*n, 2.5))
    if n == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ax.imshow(imgs[i].permute(1,2,0))
        ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_batch(train_loader)

## 5 · Model Definitions

In [ ]:
def build_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

def build_efficientnet_b3():
    model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES)
    return model

def build_vit_dino():
    # ViT-B/16 pretrained with DINO self-supervised objectives
    model = timm.create_model('vit_base_patch16_224.dino', pretrained=True, num_classes=NUM_CLASSES)
    return model

MODEL_REGISTRY = {}
if RUN_RESNET:       MODEL_REGISTRY['ResNet50']       = build_resnet50
if RUN_EFFICIENTNET: MODEL_REGISTRY['EfficientNet-B3'] = build_efficientnet_b3
if RUN_VIT:          MODEL_REGISTRY['ViT-DINO']        = build_vit_dino

print('Models queued:', list(MODEL_REGISTRY.keys()))

## 6 · Training Loop

In [ ]:
def train_model(name, model, train_loader, val_loader, epochs, lr, wd, patience):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    best_auc    = 0.0
    best_weights = None
    no_improve  = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # Validate
        model.eval()
        val_loss, correct = 0.0, 0
        all_labels, all_probs = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item() * imgs.size(0)
                probs = torch.softmax(logits, dim=1)[:, 1]
                preds = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_acc   = correct / len(val_loader.dataset)
        val_auc   = roc_auc_score(all_labels, all_probs)

        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)

        print(f'[{name}] Epoch {epoch:02d}/{epochs} '
              f'| train_loss {train_loss:.4f} '
              f'| val_loss {val_loss:.4f} '
              f'| val_acc {val_acc:.4f} '
              f'| val_AUC {val_auc:.4f}')

        # Early stopping & checkpoint
        if val_auc > best_auc:
            best_auc = val_auc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, os.path.join(OUT_DIR, f'{name}_best.pth'))
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}.')
                break

    model.load_state_dict(best_weights)
    print(f'  Best val AUC: {best_auc:.4f}')
    return model, history

In [ ]:
# Run training for all selected models
trained_models = {}
histories      = {}

for name, builder in MODEL_REGISTRY.items():
    print(f'\n{'='*55}')
    print(f'  Training: {name}')
    print(f'{'='*55}')
    #Header code

    model = builder()
    trained_models[name], histories[name] = train_model(
        name, model, train_loader, val_loader,
        EPOCHS, LR, WEIGHT_DECAY, PATIENCE
    )

## 7 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#00B4D8', '#F77F00', '#4CAF50']

for (name, hist), color in zip(histories.items(), colors):
    ep = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(ep, hist['train_loss'], label=name, color=color)
    axes[1].plot(ep, hist['val_acc'],   label=name, color=color)
    axes[2].plot(ep, hist['val_auc'],   label=name, color=color)

axes[0].set_title('Train Loss');   axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch')
axes[2].set_title('Val AUC-ROC'); axes[2].set_xlabel('Epoch')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 8 · Test Set Evaluation

In [ ]:
def evaluate(name, model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds  = logits.argmax(dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    auc = roc_auc_score(all_labels, all_probs)
    f1  = f1_score(all_labels, all_preds, average='macro')
    cm  = confusion_matrix(all_labels, all_preds)

    print(f'\n {name} Test Results')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  AUC-ROC  : {auc:.4f}')
    print(f'  F1 (macro): {f1:.4f}')
    print(classification_report(all_labels, all_preds,
                                target_names=CLASS_NAMES))
    return {'acc': acc, 'auc': auc, 'f1': f1,
            'cm': cm, 'labels': all_labels, 'probs': all_probs}

results = {name: evaluate(name, model, test_loader)
           for name, model in trained_models.items()}

In [ ]:
# Confusion matrices
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{name}\nAUC={res["auc"]:.3f}  F1={res["f1"]:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

In [ ]:
# ROC curves (all models on one plot)
plt.figure(figsize=(6, 5))
colors = ['#00B4D8', '#F77F00', '#4CAF50']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(res['labels'], res['probs'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color, lw=2)
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Test Set')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'roc_curves.png'), dpi=150)
plt.show()

## 9 · Grad-CAM (CNNs)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def run_gradcam(name, model, dataset, n_images=7):
    """Show Grad-CAM for n_images random test samples."""
    # Pick target layer per architecture
    if 'ResNet' in name:
        target_layers = [model.layer4[-1]]
    elif 'EfficientNet' in name:
        target_layers = [model.conv_head]
    else:
        print(f'Grad-CAM not applicable to {name} (use attention rollout below)')
        return

    cam = GradCAM(model=model, target_layers=target_layers)

    indices = random.sample(range(len(dataset)), n_images)
    fig, axes = plt.subplots(2, n_images, figsize=(3*n_images, 6))

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for col, idx in enumerate(indices):
        tensor, label = dataset[idx]
        input_tensor = tensor.unsqueeze(0).to(DEVICE)

        # Raw image (denormalized)
        raw = (tensor.permute(1,2,0).numpy() * std + mean).clip(0, 1)

        # Grad-CAM
        grayscale_cam = cam(input_tensor=input_tensor,
                            targets=[ClassifierOutputTarget(1)])[0]
        cam_image = show_cam_on_image(raw.astype(np.float32), grayscale_cam, use_rgb=True)

        axes[0, col].imshow(raw)
        axes[0, col].set_title(CLASS_NAMES[label], fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(cam_image)
        axes[1, col].set_title('Grad-CAM', fontsize=8)
        axes[1, col].axis('off')

    plt.suptitle(f'Grad-CAM — {name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'gradcam_{name}.png'), dpi=150)
    plt.show()

for name, model in trained_models.items():
    run_gradcam(name, model, test_ds)

## 10 · Attention Rollout (ViT-DINO)

In [ ]:
import torch.nn.functional as F
from types import MethodType

def get_attention_rollout(model, img_tensor):
    """
    Compute attention rollout for a timm ViT.
    Returns a (H, W) heatmap normalized to [0, 1].
    """
    attentions = []
    original_forward_methods = []

    # Temporarily replace the forward method of each Attention module
    # to capture the attention matrix directly.
    def custom_attention_forward(self, x, attn_mask=None, is_causal=False): # Added attn_mask and is_causal
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        # Capture the attention matrix BEFORE dropout
        attentions.append(attn.detach())

        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    for block in model.blocks:
        # Store original forward method to restore later
        original_forward_methods.append((block.attn, block.attn.forward))
        # Replace with custom forward
        block.attn.forward = MethodType(custom_attention_forward, block.attn)

    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)

    # Restore original forward methods
    for module, original_method in original_forward_methods:
        module.forward = original_method

    if not attentions:
        raise RuntimeError("No attention matrices captured after modifying forward methods. This should not happen.")

    # Rollout: multiply attention matrices across layers
    # result will be of shape (N, N) where N is num_tokens
    # attentions[0] shape is (B, heads, N_tokens, N_tokens)
    num_tokens = attentions[0].size(-1)
    result = torch.eye(num_tokens).to(DEVICE)

    for attn in attentions:
        # attn shape is (B, heads, N_tokens, N_tokens)
        attn_avg = attn.mean(dim=1)          # avg over heads, shape (B, N_tokens, N_tokens)
        # Add residual connection I for rollout as per paper
        attn_aug = attn_avg + torch.eye(num_tokens).to(DEVICE)
        attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True) # Normalize row-wise

        result = attn_aug[0] @ result

    # CLS token attention to patches
    mask = result[0, 1:]   # skip CLS token
    n_patches = int(mask.numel() ** 0.5)
    mask = mask.reshape(n_patches, n_patches).cpu().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    return mask


def show_vit_attention(name, model, dataset, n_images=6):
    if 'ViT' not in name:
        return
    indices = random.sample(range(len(dataset)), n_images)
    fig, axes = plt.subplots(2, n_images, figsize=(3*n_images, 6))
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for col, idx in enumerate(indices):
        tensor, label = dataset[idx]
        input_tensor = tensor.unsqueeze(0).to(DEVICE)
        raw = (tensor.permute(1,2,0).numpy() * std + mean).clip(0, 1)

        rollout = get_attention_rollout(model, input_tensor)
        rollout_resized = np.array(
            Image.fromarray((rollout * 255).astype(np.uint8))
                 .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        ) / 255.0

        # Overlay on raw image
        overlay = raw.copy()
        overlay[:, :, 0] = np.clip(raw[:,:,0] + 0.5 * rollout_resized, 0, 1)

        axes[0, col].imshow(raw)
        axes[0, col].set_title(CLASS_NAMES[label], fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(overlay)
        axes[1, col].set_title('Attn Rollout', fontsize=8)
        axes[1, col].axis('off')

    plt.suptitle(f'Attention Rollout — {name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'attention_{name}.png'), dpi=150)
    plt.show()


for name, model in trained_models.items():
    show_vit_attention(name, model, test_ds)

## 11 · Summary Table

In [ ]:
import pandas as pd

rows = []
for name, res in results.items():
    rows.append({
        'Model':    name,
        'Accuracy': f"{res['acc']:.4f}",
        'AUC-ROC':  f"{res['auc']:.4f}",
        'F1 (macro)': f"{res['f1']:.4f}",
    })

df = pd.DataFrame(rows).set_index('Model')
print('\nTest Set Summary')
print(df.to_string())
df.to_csv(os.path.join(OUT_DIR, 'results_summary.csv'))
print(f'\nAll outputs saved to: {OUT_DIR}')